In [ ]:
!pip install ultralytics opencv-python deep-sort-realtime torch torchvision scipy torchreid gradio pandas

import cv2
import numpy as np
import torch
import torchvision.transforms as T
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
from scipy.spatial.distance import cosine
from collections import deque, defaultdict
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
import torchreid
from google.colab.patches import cv2_imshow
from google.colab import drive
import os

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.7/92.7 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 17.1 MB/s eta 0:00:00
  Created wheel for torchreid: filename=torchreid-0.2.5-py3-none-any.whl size=144324 sha256=99017ca14d26e83c495ea93582752401fb997d5f67a246308396b6d7e0f1b477
  Stored in directory: /root/.cache/pip/wheels/5c/86/ff/80a1b78a90df470cbb12c075bf189ad33f1a41a881cf9e9a09
Successfully built torchreid
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/usr/local/lib/python3.12/dist-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


In [ ]:
@dataclass
class Config:
    environment: str = "office"
    sim_threshold: float = 0.65
    det_conf_threshold: float = 0.50
    max_track_age: int = 30
    max_embeddings: int = 10
    history_len: int = 10
    movement_threshold: float = 10.0
    stationary_threshold: float = 5.0
    elbow_angle_bent: float = 120.0
    elbow_angle_active: float = 130.0
    looking_dist_threshold: float = 150.0
    nearby_person_dist: float = 150.0
    zone_iou_threshold: float = 0.2
    hip_knee_sit_threshold: float = 0.08
    keypoint_conf_threshold: float = 0.5
    EMA_ALPHA = 0.1
    frames_dir: str = "/content/drive/MyDrive/bitirme/datasets/day_1"
    frame_sample_interval: int = 50
    zones: Dict[str, List[int]] = field(default_factory=lambda: {
        "cashier": [500, 100, 900, 600]
    })
    # ROI filtresi: [x1, y1, x2, y2] piksel koordinatlari, None = tum kare
    roi: Optional[List[int]] = None

CFG = Config()

In [ ]:
drive.mount('/content/drive')

pose_model  = YOLO('yolov8n-pose.pt')
object_model = YOLO('yolov8n.pt')
tracker     = DeepSort(max_age=CFG.max_track_age)

reid_model = torchreid.models.build_model(
    name="osnet_x1_0",
    num_classes=1000,
    pretrained=True
)
reid_model.eval()

transform = T.Compose([
    T.ToPILImage(),
    T.Resize((256, 128)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225])
])

Mounted at /content/drive


Downloading...
From: https://drive.google.com/uc?id=1LaG1EJpHrxdAxKnSCJ_i0u-nbxSAeiFY
To: /root/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth
100%|██████████| 10.9M/10.9M [00:00<00:00, 67.0MB/s]


Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"


In [ ]:
person_db: Dict[int, deque] = {}           # pid -> deque of embeddings
track_to_person: Dict[int, int] = {}       # track_id -> pid
next_person_id: int = 1
person_stats: Dict[int, Dict[str, int]] = defaultdict(lambda: defaultdict(int))
track_history: Dict[int, deque] = {}       # track_id -> deque of (cx,cy)

In [ ]:
def iou(a: List[float], b: List[float]) -> float:
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = (a[2]-a[0]) * (a[3]-a[1])
    area_b = (b[2]-b[0]) * (b[3]-b[1])
    return inter / (area_a + area_b - inter + 1e-6)

def in_zone(box: List[float], zone: List[int]) -> bool:
    return iou(box, zone) > CFG.zone_iou_threshold

def person_in_roi(box: List[float], roi: Optional[List[int]]) -> bool:
    """Kisi merkezi secili ROI icinde mi? roi=None ise herkesi kabul et."""
    if roi is None:
        return True
    cx = (box[0] + box[2]) / 2
    cy = (box[1] + box[3]) / 2
    return roi[0] <= cx <= roi[2] and roi[1] <= cy <= roi[3]

def apply_roi_dim(annotated: np.ndarray, roi: Optional[List[int]]) -> np.ndarray:
    """ROI disini %65 karartir, siniri turuncu dikdortgenle belirtir."""
    if roi is None:
        return annotated
    rx1, ry1, rx2, ry2 = roi
    dark = annotated.copy()
    dark[:, :] = (dark[:, :] * 0.35).astype(np.uint8)
    dark[ry1:ry2, rx1:rx2] = annotated[ry1:ry2, rx1:rx2]
    cv2.rectangle(dark, (rx1, ry1), (rx2, ry2), (0, 165, 255), 2)
    cv2.putText(dark, "ROI", (rx1 + 4, max(ry1 + 18, 18)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 165, 255), 2)
    return dark

def get_movement(track_id: int, cx: int, cy: int) -> float:
    if track_id not in track_history:
        track_history[track_id] = deque(maxlen=CFG.history_len)
    track_history[track_id].append((cx, cy))
    if len(track_history[track_id]) < 2:
        return 0.0
    prev = track_history[track_id][-2]
    curr = track_history[track_id][-1]
    return float(np.linalg.norm(np.array(curr) - np.array(prev)))

def elbow_angle(kp: np.ndarray, kp_conf: Optional[np.ndarray] = None) -> float:
    """Her iki dirsek acisini hesaplar, kucugunu doner (daha buk taraf).
    Bir taraf guvenilmezse digeri kullanilir; her ikisi de guvenilmezse 180 doner."""
    def _one_side(s_i, e_i, w_i):
        if kp_conf is not None and any(kp_conf[i] < CFG.keypoint_conf_threshold for i in [s_i, e_i, w_i]):
            return 180.0
        s, e, w = kp[s_i], kp[e_i], kp[w_i]
        v1 = s - e
        v2 = w - e
        norm = np.linalg.norm(v1) * np.linalg.norm(v2)
        if norm < 1e-6:
            return 180.0
        cos_val = np.clip(np.dot(v1, v2) / norm, -1.0, 1.0)
        return float(np.degrees(np.arccos(cos_val)))
    return min(_one_side(5, 7, 9), _one_side(6, 8, 10))

def looking_score(kp: np.ndarray, computers: List, w: int, h: int) -> float:
    """Distance from nose to nearest computer. Returns inf if no computers detected."""
    if not computers:
        return float('inf')
    nose = kp[0]
    nx, ny = int(nose[0] * w), int(nose[1] * h)
    best = float('inf')
    for box in computers:
        x1, y1, x2, y2 = map(int, box)
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        d = np.sqrt((nx - cx)**2 + (ny - cy)**2)
        best = min(best, d)
    return best

def nearby_person(box: List[float], all_boxes: np.ndarray) -> bool:
    px1, py1, px2, py2 = box
    pcx, pcy = (px1 + px2) // 2, (py1 + py2) // 2
    for b in all_boxes:
        x1, y1, x2, y2 = b
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        d = np.sqrt((pcx - cx)**2 + (pcy - cy)**2)
        if 30 < d < CFG.nearby_person_dist:
            return True
    return False

In [ ]:
def get_embedding(img: np.ndarray) -> Optional[np.ndarray]:
    if img is None or img.size == 0:
        return None
    try:
        tensor = transform(img).unsqueeze(0)
        with torch.no_grad():
            emb = reid_model(tensor)
        emb = emb.cpu().numpy().flatten()
        norm = np.linalg.norm(emb)
        return emb / (norm + 1e-6)
    except Exception as e:
        print(f"[ReID] Embedding error: {e}")
        return None

def match_person(emb: np.ndarray) -> int:
    global next_person_id
    best_id, best_score = None, 0.0

    for pid, emb_list in person_db.items():
        ref = np.mean(emb_list, axis=0)
        ref /= (np.linalg.norm(ref) + 1e-6)
        sim = 1.0 - cosine(emb, ref)
        if sim > best_score:
            best_score, best_id = sim, pid

    if best_score > CFG.sim_threshold and best_id is not None:
        person_db[best_id].append(emb)
        return best_id

    # new person
    person_db[next_person_id] = deque([emb], maxlen=CFG.max_embeddings)
    pid = next_person_id
    next_person_id += 1
    return pid

In [ ]:
def determine_status(env: str, f: dict) -> str:
    # Telefon: global override
    if f["looking_down"] and f["hand_up"]:
        return "Telefon"

    if env == "office":
        if f["is_sitting"] and f["arm_bent"] and (f["looking_pc"] or f["looking_down"]):
            return "Calisiyor"
        elif f["is_sitting"]:
            return "Oturuyor"
        elif f["movement"] > CFG.movement_threshold:
            return "Dolaniyor"
        else:
            return "Ayakta"

    elif env == "cafe":
        cashier_active = f["arm_active"] or f["looking_pc"] or f["wrists_at_counter"]
        if f["in_cashier"] and f["stationary"] and cashier_active:
            return "Kasiyer"
        elif f["in_cashier"] and f["stationary"]:
            return "Kasada Bos"
        elif f["movement"] > CFG.movement_threshold:
            return "Servis"
        elif f["arm_active"] or f["wrists_at_counter"]:
            return "Hazirlik"
        else:
            return "Bos"

    elif env == "market":
        if f["in_cashier"] and f["stationary"] and not f["is_sitting"] and (
            (f["wrists_at_counter"] and f["looking_down"]) or
            (f["looking_pc"]) or
            (f["arm_bent"] and f["wrists_at_counter"])
        ):
            return "Kasiyer"
        elif f["in_cashier"] and f["stationary"] and not f["is_sitting"]:
            return "Kasada Bos"
        elif f["movement"] > CFG.movement_threshold and f["arm_active"]:
            return "Servis"
        elif f["movement"] > CFG.movement_threshold:
            return "Dolaniyor"
        elif f["arm_active"] or f["wrists_at_counter"]:
            return "Hazirlaniyor"
        else:
            return "Bos"

    return "Unknown"

In [ ]:
def analyze_frame(path: str) -> Optional[np.ndarray]:
    """
    Verilen frame'i analiz eder, etiketli BGR goruntu doner.
    person_stats guncellemesi burada yapilir.
    Hata/bos durumunda None doner.
    """
    img = cv2.imread(path)
    if img is None:
        print(f"[WARN] Could not read frame: {path}")
        return None

    h, w, _ = img.shape

    if CFG.roi is not None:
        rx1, ry1, rx2, ry2 = CFG.roi
        img_roi = img[ry1:ry2, rx1:rx2]
        crop_h, crop_w = img_roi.shape[:2]
    else:
        rx1, ry1, rx2, ry2 = 0, 0, w, h
        img_roi = img
        crop_h, crop_w = h, w

    try:
        pose_result = pose_model(img_roi, verbose=False)[0]
    except Exception as e:
        print(f"[ERROR] Pose model failed on {path}: {e}")
        return None

    # YOLO'nun kendi iskelet+kutu cizimini krop uzerinde al
    annotated_crop = pose_result.plot()

    # Tam goruntu: dis alan karti, ROI'ye YOLO cizimi yapistir
    annotated = apply_roi_dim(img.copy(), CFG.roi)
    annotated[ry1:ry2, rx1:rx2] = annotated_crop

    if pose_result.boxes is None or len(pose_result.boxes) == 0:
        return annotated

    boxes_all     = pose_result.boxes.xyxy.cpu().numpy()
    confs_all     = pose_result.boxes.conf.cpu().numpy()
    kp_conf_all   = pose_result.keypoints.conf.cpu().numpy() \
                    if pose_result.keypoints.conf is not None else None
    keypoints_all = pose_result.keypoints.xyn.cpu().numpy()

    mask          = confs_all >= CFG.det_conf_threshold
    boxes         = boxes_all[mask]
    confs         = confs_all[mask]
    keypoints_all = keypoints_all[mask]
    if kp_conf_all is not None:
        kp_conf_all = kp_conf_all[mask]

    if len(boxes) == 0:
        return annotated

    if CFG.roi is not None:
        boxes[:, [0, 2]] += rx1
        boxes[:, [1, 3]] += ry1
        keypoints_all[:, :, 0] = keypoints_all[:, :, 0] * (crop_w / w) + rx1 / w
        keypoints_all[:, :, 1] = keypoints_all[:, :, 1] * (crop_h / h) + ry1 / h

    try:
        obj_result = object_model(img, verbose=False)[0]
        computer_boxes = [
            box.cpu().numpy()
            for box, cls in zip(obj_result.boxes.xyxy, obj_result.boxes.cls)
            if object_model.names[int(cls)] in {"laptop", "tv", "monitor"}
        ]
    except Exception as e:
        print(f"[WARN] Object detection failed: {e}")
        computer_boxes = []

    detections = [
        ([b[0], b[1], b[2] - b[0], b[3] - b[1]], float(confs[i]), "person")
        for i, b in enumerate(boxes)
    ]
    tracks = tracker.update_tracks(detections, frame=img)

    active_ids = {t.track_id for t in tracks if t.is_confirmed()}
    for dead_id in [tid for tid in list(track_to_person) if tid not in active_ids]:
        del track_to_person[dead_id]

    roi_count = 0
    for t in tracks:
        if not t.is_confirmed():
            continue
        if t.time_since_update > 0:
            continue

        x1, y1, x2, y2 = map(int, t.to_ltrb())
        box = [x1, y1, x2, y2]

        if not person_in_roi(box, CFG.roi):
            continue

        roi_count += 1
        best_i = max(range(len(boxes)), key=lambda i: iou(box, boxes[i]), default=-1)
        if best_i < 0:
            continue

        kp      = keypoints_all[best_i]
        kp_conf = kp_conf_all[best_i] if kp_conf_all is not None else None

        crop = img[max(0,y1):max(0,y2), max(0,x1):max(0,x2)]
        emb  = get_embedding(crop)
        if emb is None:
            continue

        if t.track_id not in track_to_person:
            pid = match_person(emb)
            track_to_person[t.track_id] = pid
        else:
            pid = track_to_person[t.track_id]
            person_db[pid].append(emb)

        cx, cy   = (x1 + x2) // 2, (y1 + y2) // 2
        movement = get_movement(t.track_id, cx, cy)

        # Oturma: sol VEYA sag kalca-diz cifti yeterliyse dogru etiketler
        def _side_ok(*idxs):
            return kp_conf is None or all(kp_conf[i] >= CFG.keypoint_conf_threshold for i in idxs)
        is_sitting = (
            (_side_ok(11, 13) and abs(kp[11][1] - kp[13][1]) < CFG.hip_knee_sit_threshold) or
            (_side_ok(12, 14) and abs(kp[12][1] - kp[14][1]) < CFG.hip_knee_sit_threshold)
        )

        angle = elbow_angle(kp, kp_conf)

        _conf = kp_conf if kp_conf is not None else np.ones(17)
        _thr  = CFG.keypoint_conf_threshold
        # wrists_at_counter: omuz ve kalca referansi icin guvenilir tarafi kullan
        _sh_y = kp[5][1] if _conf[5] >= _thr else (kp[6][1] if _conf[6] >= _thr else None)
        _hp_y = kp[11][1] if _conf[11] >= _thr else (kp[12][1] if _conf[12] >= _thr else None)
        wrists_at_counter = (
            _sh_y is not None and _hp_y is not None and (
                (_conf[9]  >= _thr and _sh_y < kp[9][1]  < _hp_y + 0.1) or
                (_conf[10] >= _thr and _sh_y < kp[10][1] < _hp_y + 0.1)
            )
        )

        features = {
            "is_sitting":        is_sitting,
            "arm_bent":          angle < CFG.elbow_angle_bent,
            "arm_active":        angle < CFG.elbow_angle_active,
            # looking_down / hand_up: her iki omuz ve bilek kontrol edilir
            "looking_down":      kp[0][1] > min(kp[5][1], kp[6][1]),
            "hand_up":           min(kp[9][1], kp[10][1]) < max(kp[5][1], kp[6][1]),
            "looking_pc":        looking_score(kp, computer_boxes, w, h) < CFG.looking_dist_threshold,
            "movement":          movement,
            "stationary":        movement < CFG.stationary_threshold,
            "in_cashier":        in_zone(box, CFG.zones["cashier"]),
            "near_person":       nearby_person(box, boxes),
            "wrists_at_counter": wrists_at_counter,
        }

        status = determine_status(CFG.environment, features)
        person_stats[pid][status] += 1

        cv2.putText(annotated, f"P{pid} | {status}",
                    (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    roi_label = f"Bolgedeki kisi: {roi_count}" if CFG.roi is not None else f"Toplam kisi: {roi_count}"
    cv2.putText(annotated, roi_label, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    return annotated

In [ ]:
frame_files = sorted(
    f for f in os.listdir(CFG.frames_dir)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
)

for i, fname in enumerate(frame_files):
    if i % CFG.frame_sample_interval == 0:
        ann = analyze_frame(os.path.join(CFG.frames_dir, fname))
        if ann is not None:
            cv2_imshow(ann)

In [ ]:
print("\n" + "="*50)
print("  ACTIVITY REPORT")
print("="*50)

for pid, stats in sorted(person_stats.items()):
    total = sum(stats.values())
    print(f"\n  Person {pid}  ({total} frames sampled)")
    print("  " + "-"*30)
    for status, count in sorted(stats.items(), key=lambda x: -x[1]):
        pct = (count / total) * 100
        bar = "█" * int(pct / 5)
        print(f"  {status:<16} {bar:<20} {pct:5.1f}%")

print("\n" + "="*50)

In [ ]:
import gradio as gr
import pandas as pd
import threading

def analyze_frame_ui(path: str):
    """analyze_frame'i cagirip BGR->RGB donusumu yapar (Gradio icin)."""
    ann = analyze_frame(path)
    if ann is None:
        return None
    return cv2.cvtColor(ann, cv2.COLOR_BGR2RGB)


def make_report_df():
    rows = []
    for pid, stats in sorted(person_stats.items()):
        total = sum(stats.values())
        for status, count in sorted(stats.items(), key=lambda x: -x[1]):
            rows.append({
                "Kisi":         f"P{pid}",
                "Aktivite":     status,
                "Frame Sayisi": count,
                "Oran (%)":     round((count / total) * 100, 1),
            })
    if not rows:
        return pd.DataFrame(columns=["Kisi", "Aktivite", "Frame Sayisi", "Oran (%)"])
    return pd.DataFrame(rows)


_state = {
    "frame":   None,
    "report":  pd.DataFrame(columns=["Kisi", "Aktivite", "Frame Sayisi", "Oran (%)"]),
    "status":  "Hazir.",
    "running": False,
    "stop":    False,
}


def _background_run(environment, frames_dir, frame_interval, sim_thr, hip_knee_thr,
                    roi_enabled, roi_x1, roi_y1, roi_x2, roi_y2):
    global person_db, track_to_person, next_person_id, person_stats, track_history

    person_db       = {}
    track_to_person = {}
    next_person_id  = 1
    person_stats    = defaultdict(lambda: defaultdict(int))
    track_history   = {}

    CFG.environment            = environment
    CFG.frames_dir             = frames_dir
    CFG.frame_sample_interval  = int(frame_interval)
    CFG.sim_threshold          = float(sim_thr)
    CFG.hip_knee_sit_threshold = float(hip_knee_thr)

    try:
        all_frames = sorted(
            f for f in os.listdir(frames_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        )
    except FileNotFoundError:
        _state["status"]  = f"Dizin bulunamadi: {frames_dir}"
        _state["running"] = False
        return

    if bool(roi_enabled) and all_frames:
        sample = cv2.imread(os.path.join(frames_dir, all_frames[0]))
        if sample is not None:
            fh, fw = sample.shape[:2]
            CFG.roi = [
                int(roi_x1 / 100 * fw),
                int(roi_y1 / 100 * fh),
                int(roi_x2 / 100 * fw),
                int(roi_y2 / 100 * fh),
            ]
        else:
            CFG.roi = None
    else:
        CFG.roi = None

    to_process = all_frames[::int(frame_interval)]
    total      = len(to_process)

    if total == 0:
        _state["status"]  = "Islenecek frame bulunamadi."
        _state["running"] = False
        return

    for idx, fname in enumerate(to_process):
        if _state["stop"]:
            _state["status"] = f"Durduruldu. ({idx}/{total} frame islendi)"
            break

        _state["status"] = f"Isleniyor: {idx+1}/{total} — {fname}"
        ann = analyze_frame_ui(os.path.join(frames_dir, fname))
        if ann is not None:
            _state["frame"] = ann
        _state["report"] = make_report_df()
    else:
        _state["status"] = f"Tamamlandi. {total} frame analiz edildi."

    _state["running"] = False
    _state["stop"]    = False


def start_analysis(environment, frames_dir, frame_interval, sim_thr, hip_knee_thr,
                   roi_enabled, roi_x1, roi_y1, roi_x2, roi_y2):
    if _state["running"]:
        return
    _state["running"] = True
    _state["stop"]    = False
    _state["status"]  = "Baslatiliyor..."
    threading.Thread(
        target=_background_run,
        args=(environment, frames_dir, frame_interval, sim_thr, hip_knee_thr,
              roi_enabled, roi_x1, roi_y1, roi_x2, roi_y2),
        daemon=True,
    ).start()


def stop_analysis():
    if _state["running"]:
        _state["stop"] = True


def poll():
    return _state["frame"], _state["report"], _state["status"]


with gr.Blocks(title="Aktivite Tanima", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Kisi Aktivite Tanima Sistemi")

    with gr.Row():
        with gr.Column(scale=1, min_width=320):
            gr.Markdown("### Konfigurasyon")
            env_dd      = gr.Dropdown(choices=["office", "cafe", "market"],
                                      value=CFG.environment, label="Ortam")
            dir_tb      = gr.Textbox(value=CFG.frames_dir,
                                     label="Frames Dizini (Drive yolu)")
            interval_sl = gr.Slider(1, 100, value=CFG.frame_sample_interval,
                                    step=1, label="Her N. Frame Islensin")
            sim_sl      = gr.Slider(0.5, 1.0, value=CFG.sim_threshold,
                                    step=0.01, label="ReID Benzerlik Esigi")
            hip_sl      = gr.Slider(0.01, 0.3, value=CFG.hip_knee_sit_threshold,
                                    step=0.005, label="Oturma Esigi (hip-knee)")

            gr.Markdown("---")
            gr.Markdown("### Bolge Secimi (ROI)")
            gr.Markdown("*Goruntu boyutunun yuzdesi (0–100). Sadece secilen bolge modele verilir.*")
            roi_chk = gr.Checkbox(label="ROI Filtresi Aktif", value=False)
            with gr.Row():
                roi_x1_sl = gr.Slider(0, 100, value=0,   step=1, label="Sol %")
                roi_y1_sl = gr.Slider(0, 100, value=0,   step=1, label="Ust %")
            with gr.Row():
                roi_x2_sl = gr.Slider(0, 100, value=100, step=1, label="Sag %")
                roi_y2_sl = gr.Slider(0, 100, value=100, step=1, label="Alt %")

            with gr.Row():
                run_btn  = gr.Button("Analizi Baslat", variant="primary")
                stop_btn = gr.Button("Durdur", variant="stop")
            status_md = gr.Markdown("Hazir.")

        with gr.Column(scale=2):
            gr.Markdown("### Anlik Frame")
            frame_img = gr.Image(label="", show_label=False, height=480)

    gr.Markdown("### Aktivite Raporu")
    report_tbl = gr.DataFrame(
        headers=["Kisi", "Aktivite", "Frame Sayisi", "Oran (%)"],
        datatype=["str", "str", "number", "number"],
        interactive=False,
    )

    timer = gr.Timer(value=0.5)

    run_btn.click(
        fn=start_analysis,
        inputs=[env_dd, dir_tb, interval_sl, sim_sl, hip_sl,
                roi_chk, roi_x1_sl, roi_y1_sl, roi_x2_sl, roi_y2_sl],
        outputs=[],
    )

    stop_btn.click(fn=stop_analysis, inputs=[], outputs=[])

    timer.tick(fn=poll, outputs=[frame_img, report_tbl, status_md])

demo.launch(share=True, debug=False)